## My Orbital Mechanics Walkthrough using python. 

You'll note that all the vector math is done manually or implemented in this file to show how the vector math works.

I created this to help learn how orbital mechanics and the math behind it works.

In [1]:
import numpy as np # linear algebra
import math 
import os

## Vector Math Functions!!
def calc_magnitude(vector):
    return math.sqrt(pow(vector[0],2)+pow(vector[1],2)+pow(vector[2],2));

def calc_dot_prod(vector1, vector2):
    return (vector1[0]*vector2[0])+(vector1[1]*vector2[1])+(vector1[2]*vector2[2])

def cross_product(vector1, vector2):
    prod = [0,0,0]
    prod[0] = (vector1[1]*vector2[2])-(vector2[1]*vector1[2])
    prod[1] = -1*((vector1[0]*vector2[2])-(vector2[0]*vector1[2]))
    prod[2] = (vector1[0]*vector2[1])-(vector2[0]*vector1[1])
    return prod

def divide_vec(vec, divisor):
    return [vec[0]/divisor,vec[1]/divisor,vec[2]/divisor]

def sub_vec(vec1, vec2):
    return [vec1[0]-vec2[0],vec1[1]-vec2[1],vec1[2]-vec2[2]]

## Not vector math but useful for later on
def quadrant_check(check_val, angle):
    if(check_val < 0):
        return 360-angle
    return angle

## Initial Parameters

We are going to define our initial parameters in this section. R represents the position of the spacecraft, and V represents it's velocity. Both vectors are relative to the earth's inertial frame, which is defined by the earth's equitorial plane and the north pole vector.

Feel free to mess around with these values to see how it will impact the orbit!

In [2]:
# R = [-6045, -3490, 2500]
# V = [-3.457, 6.618, 2.533]
#R = [881.4030532, -32238.9264, 1280.93121]
#V = [2.576056065, -3.499073389, -1.3146739435]
R = [14429.481,-47206.152,-5824.773]
V = [2.362,-2.185,-1.262]


## Calculate Magnitude of position and velocity for later

In [3]:
mag_R = calc_magnitude(R)
mag_V = calc_magnitude(V)

print(f'|R|={mag_R} (km), |V|={mag_V} (km/s)')

|R|=49704.71495822096 (km), |V|=3.4562860124706116 (km/s)


## Calculate Unit Vectors for R & V (for later)
A unit vector tells us direction, without magnitude, and is useful for operations that don't require scale, and are generally easier to work with when doing reference frame transformations and rotations. The unit vector also gives us a general idea of the proportions of the vector components we're working with, relative to eachother.

In [4]:
unit_R = divide_vec(R, mag_R)
unit_V = divide_vec(V, mag_V)

print(f'unit_R={unit_R} (km), unit_V={unit_V} (km/s)')

unit_R=[0.290304068982764, -0.94973187231189, -0.1171875345205376] (km), unit_V=[0.6833925177134292, -0.6321814780710596, -0.36513181937101935] (km/s)


## Dot product of position and velocity for later

In [5]:
R_dot_V = calc_dot_prod(R, V)

print(f'R*V={R_dot_V} (km^2/s)')

R*V=144578.739768 (km^2/s)


# Find Angular Momentum

Momentum is the cross product of the position and velocity vectors:

> H = R ✕ V

It can help us define the orbital plane (together with the position vector)
We also calculate the maginitude of this vector for later use.

In [6]:
H = cross_product(R,V)
mag_H = calc_magnitude(H)
unit_H = divide_vec(H,mag_H)

print(f'H=RxV={H}')
print(f'|H|={mag_H} (km^2/sec)')
print(f'unit_H={unit_H}')

H=RxV=[46847.034819, 4451.8911960000005, 79972.515039]
|H|=92790.4476129774 (km^2/sec)
unit_H=[0.5048691543594636, 0.04797790408953014, 0.8618615072594529]


# Solve for Energy

Solving for energy allows us to calculate the SMA because we can use it to show the maximum altitude an orbit can reach (the furthest point!) This number should ALWAYS be negative for an eliptical orbit.

> E = |V|<sup>2</sup>/2 - mu/|R|

In [7]:
mu = 3.986*pow(10,5)

def calc_orbital_energy(mag_vel, mag_rad):
    return (pow(mag_vel,2)/2)-(mu/(mag_rad))

E = calc_orbital_energy(mag_V, mag_R)
print(f'E={E} (km^2/s^2)')

E=-2.0464034407026297 (km^2/s^2)


## Solve For Semi-major Axis (a | SMA)

The semi-major axis (the long arm of the elipse's diameter) helps us establish the shape orbit, in conjunction with eccentricity, and is needed to calculate eccentricity. This number should ALWAYS be positive for an eliptical orbit.

> a = -mu/(2\*E)

In [8]:
def get_SMA(energy):
    return (-1*mu)/(2*energy)

SMA = get_SMA(E)
print(f'SMA={SMA} (km)')

SMA=97390.3757372352 (km)


## Solve for Eccentricity (e)

Eccentricity describes how elliptical an orbit is. The closer the magnitude is to 0 the more circular, the closer to 1 the more stretched the ellipse will appear.

> e = ((V ✕ H) / mu) - (R / |R|)

Here we calculate the eccentricity vector, and it's magnitude.

The vector is find by: 
1. taking the cross product of velocity and angular momentum.
2. dividing this result by mu
3. subtracting the unit vector representing how the radius vector "points" (divide radius by it's magnitude)

In [9]:
def get_ecc_vec(vel, ang_mom, radius, mag_radius):
    vec1 = cross_product(vel,ang_mom)
    print(f'step 1: V x H = {vec1}')
    vec1_div_mu = divide_vec(vec1,mu)
    print(f'step 2: V x H / mu = {vec1_div_mu}')
    rad_div_mag = divide_vec(radius,mag_radius)
    print(f'step 2.5: radial unit vector = {rad_div_mag}')
    result = sub_vec(vec1_div_mu,rad_div_mag)
    print(f'step 3: subtract radial unit vector from  V x H /mu = {result}')
    return result

ecc = get_ecc_vec(V, H, R, mag_R)
mag_ecc = calc_magnitude(ecc)
unit_ecc = divide_vec(ecc,mag_ecc)
print(f'---------------------------------')
print(f'ecc = {ecc}')
print(f'|ecc| = {mag_ecc}')
print(f'unit_ecc = {unit_ecc}')

step 1: V x H = [-169121.658670863, -248016.03846369602, 112876.138084467]
step 2: V x H / mu = [-0.424289158732722, -0.622217858664566, 0.28318148039254143]
step 2.5: radial unit vector = [0.290304068982764, -0.94973187231189, -0.1171875345205376]
step 3: subtract radial unit vector from  V x H /mu = [-0.7145932277154861, 0.327514013647324, 0.400369014913079]
---------------------------------
ecc = [-0.7145932277154861, 0.327514013647324, 0.400369014913079]
|ecc| = 0.8821588623001445
unit_ecc = [-0.8100505002605232, 0.37126421061322523, 0.4538513775955901]


## Solve for Inclination

Inclination tells us the tilt, or difference between the orbital plane and the earth's equitorial plane.

It's calculated by taking the inverse cosine of the angular momentum unit vector's k component.

> i = cos<sup>-1</sup>(H<sub>k</sub>/|H|)

Note: because we're using python math, all trig functions operate in radians, and their results must be converted to degrees.

In [10]:
i = math.degrees(math.acos(unit_H[2]))
print(f'i={i} (deg)')

i=30.473760269987377 (deg)


## North Pole and Node vector

In order to move to the next step, we need to know the unit vector that represents the earth's north pole, relative to the equitorial plane, and the Node vector, which is the line formed by the intersection of the orbital and equitorial plane.

> N = \[0,0,1\] ✕ H

In [11]:
north_pole_vec = [0,0,1]
N = cross_product(north_pole_vec,H)
mag_N = calc_magnitude(N)
unit_N = divide_vec(N,mag_N)

print(f'Node Vector={N}')
print(f'mag_N={mag_N}')
print(f'unit_N={unit_N}')


Node Vector=[-4451.8911960000005, 46847.034819, 0.0]
mag_N=47058.09182864962
unit_N=[-0.09460415888112206, 0.9955149688088047, 0.0]


## Solve for RAAN

Right ascension of ascending node shows us how the orbit is "rotated"

Similarly to inclination we find it by taking the inverse cosine of the Node unit vector's I component.

> RAAN = cos<sup>-1</sup>(unit_N<sub>i</sub>)

But then we must do a quadrant check to determine if the angle is between 0 and 180 or 180 and 360, using the sign of the Node vector's j component.
> N<sub>j</sub> >= 0 -> 0-180
> N<sub>j</sub> < 0 -> 180-360


In [12]:
RAAN_init = math.degrees(math.acos(unit_N[0]))

RAAN = quadrant_check(N[1],RAAN_init)

print(f'RAAN={RAAN} (deg)')


RAAN=95.42853717700574 (deg)


## Solve for Argument of Periapsis

To solve for this, we need to: 
1. find the dot product of the Node and eccentricity unit vectors
2. take the inverse cosine of the above

> w = cos<sup>-1</sup>(unit_N\*unit_ecc)

3. do a quadrant check. If eccentricity's J component is positive, we're done, otherwise, if ecc_j is negative, we subtract our degree value from 360.
> - ecc<sub>j</sub> >= 0 -> 0-180
> - ecc<sub>j</sub> < 0 -> 180-360

In [13]:
N_dot_ecc = calc_dot_prod(unit_N,unit_ecc)
w_init = math.degrees(math.acos(N_dot_ecc))
w = quadrant_check(ecc[2],w_init)
print('----------------------------------------')
print(f'w={w} (deg)')

----------------------------------------
w=63.497732808330895 (deg)


## Solve for True Anomaly

True anomaly tells us where in the orbit the spacecraft is. To find it we: 
1. take the dot product of eccentricity and position unit vectors.
2. take the inverse cosine

> theta = cos<sup>-1</sup>(unit_ecc\*unit_R)

3. do a quadrant check using the dot-product of position and velocity.

> - R\*V >= 0 -> 0-180
> - R\*V < 0 -> 180-360

In [14]:
TA_init = math.degrees(math.acos(calc_dot_prod(unit_ecc,unit_R)))
TA = quadrant_check(calc_dot_prod(R,V),TA_init)
print('----------------------------------------')
print(f'TA={TA} (deg)')

----------------------------------------
TA=129.8625558604415 (deg)


## Solve for Period

The orbital period indicates how quickly the spacecraft completes one orbit.

We use the orbital equation to find it:
> T = 2\*pi/sqrt(mu)\*a<sup>3/2</sup>

In [15]:
T = 2*math.pi/math.sqrt(mu)*pow(SMA,3/2)
print(f'T={T} (s)')
print(f'T={T/60} (min)')
print(f'T={T/3600} (hr)')

T=302472.07330926537 (s)
T=5041.201221821089 (min)
T=84.02002036368482 (hr)


## Solve for Radius at various points of the ellipse

1. perogee is the closest point to the orbited body on the elipse, and where the spacecraft moves quickest.
2. appogee is the furthest point to the orbited body on the elipse, and where the spacecraft moves the slowest.

These three radii describe the altitude of the orbit at important places in the elipse. It's lowest, highest, and and current position in the ellipse.

In [16]:
R_p = (pow(mag_H,2)/mu) * (1/(1+mag_ecc))
R_a = (pow(mag_H,2)/mu) * (1/(1-mag_ecc))

c0 = math.cos(math.radians(TA))

R_TA = (pow(mag_H,2)/mu) * (1/(1+(mag_ecc*c0)))

print(f'R_p={R_p} (km)')
print(f'R_a={R_a} (km)')
print(f'R_TA={R_TA} (km)')


R_p=11476.592677892182 (km)
R_a=183304.15879657798 (km)
R_TA=49704.71495822097 (km)


In [17]:
def print_orb():
    print(f'-----------------------------------------')
    print(f'Position and Velocity Vectors')
    print(f'-----------------------------------------')
    print(f'R={R} (k), V={V} (km/s)')
    print(f'-----------------------------------------')
    print(f'Helpers')
    print(f'-----------------------------------------')
    print(f'|R|={mag_R} (km), |V|={mag_V} (km/s)')
    print(f'R*V={R_dot_V} (km^2/s)')
    print(f'Angular Momentum:H=RxV={H} (km^2/s), |H|={mag_H} (km^2/s)')
    print(f'Node Vector: N={N}, |N|={mag_N}')
    print(f'Energy in the Orbital System: E={E} (km^2/s^2)')
    print(f'-----------------------------------------')
    print(f'Shape')
    print(f'-----------------------------------------')
    print(f'Semi Major Axis: SMA={SMA} (km)')
    print(f'Eccentricity: ecc = {ecc}, |ecc| = {mag_ecc}')
    print(f'-----------------------------------------')
    print(f'Orientation')
    print(f'-----------------------------------------')
    print(f'Inclination(tilt): i={i} (deg)')
    print(f'Right Ascention of Ascending Node(rotation): RAAN={RAAN} (deg)')
    print(f'Argument of Pariapsis(twist): w={w} (deg)')
    print(f'True Anomaly(spacecraft position on ellipse): TA={TA} (deg)')
    print(f'-----------------------------------------')
    print(f'Orbital Properties')
    print(f'-----------------------------------------')    
    print(f'Orbital Period: T={T} (s)')
    print(f'Radius at Perigee, Apogee, and True Anomaly:')
    print(f' - R_p={R_p} (km)')
    print(f' - R_a={R_a} (km)')
    print(f' - R_TA={R_TA} (km)')
    
print_orb()


-----------------------------------------
Position and Velocity Vectors
-----------------------------------------
R=[14429.481, -47206.152, -5824.773] (k), V=[2.362, -2.185, -1.262] (km/s)
-----------------------------------------
Helpers
-----------------------------------------
|R|=49704.71495822096 (km), |V|=3.4562860124706116 (km/s)
R*V=144578.739768 (km^2/s)
Angular Momentum:H=RxV=[46847.034819, 4451.8911960000005, 79972.515039] (km^2/s), |H|=92790.4476129774 (km^2/s)
Node Vector: N=[-4451.8911960000005, 46847.034819, 0.0], |N|=47058.09182864962
Energy in the Orbital System: E=-2.0464034407026297 (km^2/s^2)
-----------------------------------------
Shape
-----------------------------------------
Semi Major Axis: SMA=97390.3757372352 (km)
Eccentricity: ecc = [-0.7145932277154861, 0.327514013647324, 0.400369014913079], |ecc| = 0.8821588623001445
-----------------------------------------
Orientation
-----------------------------------------
Inclination(tilt): i=30.473760269987377 (de